<div style="text-align: center; background-color: #5A96E3; font-family: 'Trebuchet MS', Arial, sans-serif; color: white; padding: 20px; font-size: 40px; font-weight: bold; border-radius: 0 0 0 0; box-shadow: 0px 6px 8px rgba(0, 0, 0, 0.2);">
  Stage 03 - Extract candidate cv 📌
</div>

## I. Import libraries

In [1]:
import pandas as pd
import fitz
import os
import json
import re
import uuid
from langchain_qdrant import Qdrant
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
# from qdrant_client.models import t, VectorParams, Distance
from langchain_deepseek import ChatDeepSeek
from langchain_community.embeddings import GPT4AllEmbeddings
from langchain.schema import HumanMessage
from langchain.schema import Document



## II. Config environment

In [ ]:
os.environ["DEEPSEEK_API_KEY"] = ""

## II. Extracting cv content from pdf format

Extrating raw text content in a CV.

In [3]:
def extract_text_from_pdf(file_path: str) -> str:
    doc = fitz.open(file_path)
    text = ""
    for page in doc:
        text += page.get_text()
    doc.close()
    return text.strip()

raw_text = extract_text_from_pdf("../cv_database/01.pdf")


Extract each field using a LLM model.

Define a LLM and a embedding model.

In [4]:
llm = ChatDeepSeek(model="deepseek-chat")
# embedding = GPT4AllEmbeddings()

Define a prompt template for extrating

In [5]:
prompt_template = """
Extract the following candidate information fields from the CV content (as plain text) below in the exact JSON format:
{{
"full_name": "...",
"email": "...",
"phone": "...",
"job_title": "...",
"education": [
    {{
    "degree": "...",
    "university": "...",
    "start_year": ...,
    "end_year": ...
    }}
],
"experience": [
    {{
    "job_title": "...",
    "company": "...",
    "start_date": "...",
    "end_date": "...",
    "description": "..."
    }}
],

"skills": ["...", "..."],
"certifications": [
    {{
    "certificate_name": "...",
    "organization": "..."
    }}
],
"languages": ["...", "..."]
}}

Only include **real work experience** (e.g. internships, jobs at companies, freelance work) in the "experience" field.  
**Do not include personal, academic, or side projects** in the experience section.

Only return the JSON content. Do not include any explanation.  
If any field cannot be found, set it to null or empty array.

CV content:
{text}
"""


In [ ]:
def extract_info(text: str) -> dict:
    prompt = prompt_template.format(text=text)
    messages = [HumanMessage(content=prompt)]
    response = llm.invoke(messages)
    raw_content = response.content

    
    cleaned_data = re.sub(r"^```json\s*|\s*```$", "", raw_content.strip(), flags=re.MULTILINE)
    # candidate_info = json.loads(response.content)
    
    try:
        candidate_info = json.loads(cleaned_data)

        # Lọc experience: bỏ các mục có company = None hoặc ""
        if "experience" in candidate_info and isinstance(candidate_info["experience"], list):
            filtered_exp = []
            for exp in candidate_info["experience"]:
                company = exp.get("company")
                if company not in [None, ""]:
                    filtered_exp.append(exp)
            candidate_info["experience"] = filtered_exp

    except Exception as e:
        print(f"Error parsing JSON: {e}\nLLM output: {cleaned_data}")
        candidate_info = {}
    return candidate_info

    

In [ ]:
def process_cvs(input_dir: str, output_file: str, limit: int = 10):
    results = []

    # Lấy danh sách file pdf 
    pdf_files = [f for f in os.listdir(input_dir) if f.lower().endswith(".pdf")]
    pdf_files = pdf_files[:limit]

    for filename in pdf_files:
        file_path = os.path.join(input_dir, filename)
        print(f"Processing {file_path}...")

        # trích xuất text từ PDF
        text = extract_text_from_pdf(file_path)

        # trích xuất thông tin từ LLM
        info = extract_info(text)

        # Thêm tên file để dễ đối chiếu
        info["source_file"] = filename
        results.append(info)

    # B3: lưu kết quả thành JSON (list of objects)
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=4)

    print(f"Saved {len(results)} CVs into {output_file}")

In [8]:
process_cvs("../cv_database", "../cv_database/json_file/candidates.json", limit=33)


Processing ../cv_database\01.pdf...
Processing ../cv_database\02.pdf...
Processing ../cv_database\03.pdf...
Processing ../cv_database\04.pdf...
Processing ../cv_database\05.pdf...
Processing ../cv_database\06.pdf...
Processing ../cv_database\07.pdf...
Processing ../cv_database\08.pdf...
Processing ../cv_database\09.pdf...
Processing ../cv_database\10.pdf...
Processing ../cv_database\11.pdf...
Processing ../cv_database\12.pdf...
Processing ../cv_database\13.pdf...
Processing ../cv_database\14.pdf...
Processing ../cv_database\15.pdf...
Processing ../cv_database\16.pdf...
Processing ../cv_database\17.pdf...
Processing ../cv_database\18.pdf...
Processing ../cv_database\19.pdf...
Processing ../cv_database\20.pdf...
Processing ../cv_database\21.pdf...
Processing ../cv_database\22.pdf...
Processing ../cv_database\23.pdf...
Processing ../cv_database\24.pdf...
Processing ../cv_database\25.pdf...
Processing ../cv_database\26.pdf...
Processing ../cv_database\27.pdf...
Processing ../cv_database\28

## III. Add to Qdrant vector db

In [3]:
collection_name = "candidates"
embedding_model = GPT4AllEmbeddings()

# client = QdrantClient(path="../qdrant_initial_db")
# client.recreate_collection(
#     collection_name=collection_name,
#     vectors_config=VectorParams(size=384, distance=Distance.COSINE)
# )

In [ ]:
# with open("../cv_database/json_file/candidates.json", "r", encoding="utf-8") as f:
#     candidates = json.load(f)

# points = []
# for cand in candidates:
#     name = cand.get("full_name", "")
#     email = cand.get("email", "")
#     skills = cand.get("skills", [])
#     exp_list = cand.get("experience", [])

#     if isinstance(skills, str):  
#         skills = [skills]   # ép thành list nếu là string
#     elif skills is None:
#         skills = []

#     # Chuẩn hóa text để embedding
#     exp_texts = [f"{e.get('job_title','')} at {e.get('company','')}" for e in exp_list]
#     exp_text = " | ".join(exp_texts)

#     text_to_embed = f"Name: {name}, Email: {email}, Skills: {', '.join(skills)}, Experience: {exp_text}"

#     #vector = embedding_model.encode(text_to_embed).tolist()
#     vector = embedding_model.embed_query(text_to_embed)
#     points.append(
#         PointStruct(
#             id=str(uuid.uuid4()),  # cải thiện Indexing
#             vector=vector,
#             payload={ #thêm metadata
#                 "name": name,
#                 "email": email,
#                 "skills": skills,
#                 "experience": exp_text,
#                 "text": text_to_embed
#             }, #tăng độ chi tiết dữ liệu được index
#         )
#     )

# # Insert vào Qdrant 
# client.upsert(collection_name=collection_name, points=points)

# print(f"Inserted {len(points)} candidates into Qdrant 🚀")


Inserted 33 candidates into Qdrant 🚀


In [ ]:
with open("../cv_database/json_file/candidates.json", "r", encoding="utf-8") as f:
    candidates = json.load(f)

all_docs = []
for cand in candidates:
    name = cand.get("full_name", "")
    email = cand.get("email", "")
    skills = cand.get("skills", [])
    exp_list = cand.get("experience", [])

    if isinstance(skills, str):  
        skills = [skills]   # ép thành list nếu là string
    elif skills is None:
        skills = []

    #  chuẩn hóa experience thành chuỗi, Enhancing data granularity (tăng độ chi tiết dữ liệu)
    exp_texts = [f"{e.get('job_title','')} at {e.get('company','')}" for e in exp_list]
    exp_text = " | ".join(exp_texts)

    text_to_embed = f"Name: {name}, Email: {email}, Skills: {', '.join(skills)}, Experience: {exp_text}"

    all_docs.append(
        Document(
            page_content=text_to_embed,
            metadata={  #thêm metadata
                "name": name,
                "email": email,
                "skills": skills,
                "experience": exp_text
            }
        )
    )
    
# collection_name = "candidates"
# embedding_model = GPT4AllEmbeddings()
    
vectorstore = Qdrant.from_documents(
    all_docs,
    embedding=embedding_model,
    collection_name=collection_name,
    path="../qdrant_db")    


In [5]:
retriever = vectorstore.as_retriever(search_kwargs={"k":6})
results = retriever.get_relevant_documents("6 Frontend React.js developers")

for doc in results:
    print(doc.metadata)
    print(doc.page_content)

{'name': 'Obi Nwokogba', 'email': 'obi.nwokogba@gmail.com', 'skills': ['JavaScript', 'TypeScript', 'Python', 'CSS3', 'HTML5', 'Java', 'PHP', 'SQL', 'Sass', 'MQL', 'Angular', 'React', 'React Native', 'HTMX', 'Express', 'NestJS', 'MongoDB', 'NodeJS', 'Responsive Design', 'Android App Development', 'UI design', 'Database Architecture', 'Git', 'Data Structures', 'Version Repository', 'Web APIs', 'Data Visualization', 'Agile Methodology', 'Front-End Web Development', 'Full-Stack Web Development', 'MySQL', 'Bootstrap', 'ERDs', 'Graphic Design', 'Algorithmic Trading', 'Financial Markets'], 'experience': 'Frontend Engineer at Madison Logic Inc. | Full Stack Engineer, Graphic Designer at Pregen Inc.', '_id': '438a7579820048b7b961abc86ffa6849', '_collection_name': 'candidates'}
Name: Obi Nwokogba, Email: obi.nwokogba@gmail.com, Skills: JavaScript, TypeScript, Python, CSS3, HTML5, Java, PHP, SQL, Sass, MQL, Angular, React, React Native, HTMX, Express, NestJS, MongoDB, NodeJS, Responsive Design, A